# Submission Blender — Predicting Student Health Risk

Object-oriented audit and blending workflow for hard-label Kaggle submissions.

## tl;dr

This notebook validates every candidate, measures agreement and diversity, neutralizes duplicate model families, and exports five transparent blends. It does not tune weights against the public leaderboard.

## Context & Methods

### Key Assumptions

- Every submission has `id` and `health_condition` columns.
- Allowed labels are `fit`, `at-risk`, and `unhealthy`.
- Scores embedded in file names or the local manifest are public leaderboard metadata, not training targets.
- Hard labels support voting, not probability averaging.
- Near-identical submissions must not receive repeated full votes.

In [1]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Iterable
import html
import math

import numpy as np
import pandas as pd
from IPython.display import HTML, display


@dataclass(frozen=True)
class BlendConfig:
    """Keeps paths, schema, labels, and blend constants visible in one place."""

    competition_slug: str = "playground-series-s6e7"
    dataset_slug: str = "health-submiss-pool"
    id_column: str = "id"
    target: str = "health_condition"
    labels: tuple[str, ...] = ("fit", "at-risk", "unhealthy")
    local_pool: Path = Path("../health-submiss-pool")
    kaggle_pool: Path = Path("/kaggle/input/health-submiss-pool")
    kaggle_output: Path = Path("/kaggle/working/blends")
    near_duplicate_threshold: float = 0.9995
    score_weight_scale: float = 200.0
    svg_width: int = 1080

    def resolve_pool(self) -> Path:
        """Finds the pool by structure instead of relying on Kaggle mount names."""

        candidates = [
            self.kaggle_pool,
            self.kaggle_pool / self.dataset_slug,
            self.local_pool,
            Path(self.dataset_slug),
        ]
        kaggle_input = Path("/kaggle/input")
        if kaggle_input.exists():
            candidates.extend(path.parent for path in kaggle_input.rglob("public"))
        valid = []
        for candidate in candidates:
            if (candidate / "public").is_dir() and (candidate / "mine").is_dir():
                resolved = candidate.resolve()
                if resolved not in valid:
                    valid.append(resolved)
        if len(valid) == 1:
            return valid[0]
        if not valid:
            visible = sorted(str(path) for path in kaggle_input.glob("**/*") if path.is_dir()) if kaggle_input.exists() else []
            raise FileNotFoundError(f"Submission pool not found. Kaggle directories: {visible[:40]}")
        raise RuntimeError(f"Multiple submission pools found: {valid}")

    def resolve_output(self, pool: Path) -> Path:
        """Uses Kaggle working storage online and the dataset blend folder locally."""

        return self.kaggle_output if Path("/kaggle/working").exists() else pool / "blends"


config = BlendConfig()

## Data

### 1. Load submission pool

The repository discovers CSV files from `public/` and `mine/`. A stable candidate name combines source and file stem, preventing collisions between equal scores.

In [2]:
class SubmissionRepository:
    """Loads candidate files and score metadata without manual file lists."""

    def __init__(self, config: BlendConfig):
        self.config = config
        self.pool = config.resolve_pool()
        self.output = config.resolve_output(self.pool)

    def _manifest_scores(self) -> dict[str, float]:
        """Maps local file basenames to their best recorded public score."""

        path = self.pool / "mine" / "manifest.csv"
        if not path.exists():
            return {}
        manifest = pd.read_csv(path)
        manifest["basename"] = manifest["file_name"].map(lambda value: Path(str(value)).name)
        return manifest.groupby("basename")["public_score"].max().astype(float).to_dict()

    @staticmethod
    def _score_from_name(path: Path) -> float:
        """Reads a leading numeric score from a public submission filename."""

        token = path.stem.split("_")[0]
        try:
            return float(token)
        except ValueError:
            return float("nan")

    def catalog(self) -> pd.DataFrame:
        """Builds one metadata row per prediction CSV."""

        mine_scores = self._manifest_scores()
        records = []
        for source in ("public", "mine"):
            for path in sorted((self.pool / source).glob("*.csv")):
                if path.name == "manifest.csv":
                    continue
                score = mine_scores.get(path.name, self._score_from_name(path))
                records.append(
                    {
                        "name": f"{source}:{path.stem}",
                        "source": source,
                        "file_name": path.name,
                        "path": path,
                        "public_score": float(score),
                    }
                )
        catalog = pd.DataFrame(records).sort_values(["source", "public_score", "name"], ascending=[True, False, True])
        if catalog.empty:
            raise ValueError("No submission CSV files found")
        return catalog.reset_index(drop=True)

    def load(self, catalog: pd.DataFrame) -> dict[str, pd.DataFrame]:
        """Loads every catalog entry as a two-column dataframe."""

        return {row["name"]: pd.read_csv(row["path"]) for _, row in catalog.iterrows()}


repository = SubmissionRepository(config)
catalog = repository.catalog()
submissions = repository.load(catalog)
catalog_view = catalog[["name", "source", "public_score"]].rename(columns={"name": "candidate", "public_score": "score"})
display(catalog_view.style.format({"score": "{:.5f}"}).hide(axis="index"))

candidate,source,score
mine:0.9509,mine,0.95090
mine:0.95088,mine,0.95088
mine:0.95006,mine,0.95006
mine:0.94991,mine,0.94991
mine:0.94974,mine,0.94974
public:0.95114,public,0.95114
public:0.95114_another,public,0.95114
public:0.95113,public,0.95113
public:0.95112,public,0.95112
public:0.95095,public,0.95095


### 2. Validate inputs

Validation stops early on schema drift, duplicate identifiers, missing labels, invalid classes, or mismatched row order. Blending invalid files would produce a plausible-looking but wrong submission.

In [3]:
class SubmissionValidator:
    """Enforces competition schema and row alignment before analysis."""

    def __init__(self, config: BlendConfig):
        self.config = config

    def validate(self, submissions: dict[str, pd.DataFrame]) -> pd.DataFrame:
        """Returns compact quality evidence and raises on unsafe inputs."""

        expected_columns = [self.config.id_column, self.config.target]
        reference_ids = None
        records = []
        for name, frame in submissions.items():
            schema_valid = frame.columns.tolist() == expected_columns
            unique_ids = frame[self.config.id_column].is_unique if schema_valid else False
            missing = int(frame.isna().sum().sum())
            invalid_labels = sorted(set(frame[self.config.target].dropna()) - set(self.config.labels)) if schema_valid else []
            ids = frame[self.config.id_column].to_numpy() if schema_valid else np.array([])
            if reference_ids is None:
                reference_ids = ids
            aligned = bool(np.array_equal(reference_ids, ids))
            records.append(
                {
                    "name": name,
                    "rows": len(frame),
                    "schema_valid": schema_valid,
                    "unique_ids": unique_ids,
                    "missing_values": missing,
                    "invalid_labels": ", ".join(invalid_labels),
                    "id_order_aligned": aligned,
                }
            )
        report = pd.DataFrame(records)
        safe = (
            report["schema_valid"]
            & report["unique_ids"]
            & report["id_order_aligned"]
            & report["missing_values"].eq(0)
            & report["invalid_labels"].eq("")
        )
        if not safe.all():
            raise ValueError(f"Unsafe submissions detected:\n{report.loc[~safe].to_string(index=False)}")
        if report["rows"].nunique() != 1:
            raise ValueError("Submission row counts do not match")
        return report


validator = SubmissionValidator(config)
quality_report = validator.validate(submissions)
quality_view = quality_report[["name", "rows"]].rename(columns={"name": "candidate"})
quality_view["status"] = "ready"
display(quality_view.style.hide(axis="index"))

candidate,rows,status
mine:0.9509,295753,ready
mine:0.95088,295753,ready
mine:0.95006,295753,ready
mine:0.94991,295753,ready
mine:0.94974,295753,ready
public:0.95114,295753,ready
public:0.95114_another,295753,ready
public:0.95113,295753,ready
public:0.95112,295753,ready
public:0.95095,295753,ready


## Results

### 3. Measure agreement and diversity

Agreement measures how often two submissions predict the same label. Diversity is one minus a candidate's mean agreement with all other candidates. It is descriptive evidence, not proof that a model is better.

In [4]:
class SubmissionAnalyzer:
    """Builds aligned predictions and high-signal diversity diagnostics."""

    def __init__(self, config: BlendConfig, catalog: pd.DataFrame, submissions: dict[str, pd.DataFrame]):
        self.config = config
        self.catalog = catalog.set_index("name")
        self.submissions = submissions
        first = next(iter(submissions.values()))
        self.ids = first[config.id_column].copy()
        self.predictions = pd.DataFrame({name: frame[config.target].to_numpy() for name, frame in submissions.items()})

    def agreement(self) -> pd.DataFrame:
        """Computes the pairwise share of equal hard-label predictions."""

        names = self.predictions.columns.tolist()
        values = self.predictions.to_numpy()
        matrix = np.eye(len(names), dtype=float)
        for left in range(len(names)):
            for right in range(left + 1, len(names)):
                value = float(np.mean(values[:, left] == values[:, right]))
                matrix[left, right] = value
                matrix[right, left] = value
        return pd.DataFrame(matrix, index=names, columns=names)

    def summary(self, agreement: pd.DataFrame) -> pd.DataFrame:
        """Combines score, class balance, agreement, and diversity."""

        counts = pd.DataFrame({name: self.predictions[name].value_counts() for name in self.predictions}).T.fillna(0)
        result = self.catalog[["source", "file_name", "public_score"]].copy()
        result["mean_agreement"] = (agreement.sum(axis=1) - 1) / max(len(agreement) - 1, 1)
        result["diversity"] = 1 - result["mean_agreement"]
        for label in self.config.labels:
            result[f"share_{label}"] = counts.get(label, 0) / len(self.predictions)
        return result.sort_values(["diversity", "public_score"], ascending=[False, False])

    def disagreement_strength(self) -> pd.Series:
        """Counts the size of the largest label coalition for each row."""

        return self.predictions.apply(lambda row: row.value_counts().max(), axis=1).value_counts().sort_index()


analyzer = SubmissionAnalyzer(config, catalog, submissions)
agreement = analyzer.agreement()
summary = analyzer.summary(agreement)
vote_strength = analyzer.disagreement_strength()
summary_view = summary[["source", "public_score", "diversity", "share_fit", "share_at-risk", "share_unhealthy"]].rename(columns={"public_score": "score"})
display(summary_view.style.format({"score": "{:.5f}", "diversity": "{:.3%}", "share_fit": "{:.2%}", "share_at-risk": "{:.2%}", "share_unhealthy": "{:.2%}"}))

,source,score,diversity,share_fit,share_at-risk,share_unhealthy
name,,,,,,
mine:0.9509,mine,0.95090,0.451%,7.38%,81.15%,11.47%
mine:0.95006,mine,0.95006,0.426%,7.39%,80.94%,11.68%
mine:0.94974,mine,0.94974,0.410%,7.35%,81.15%,11.50%
mine:0.94991,mine,0.94991,0.398%,7.37%,81.01%,11.62%
mine:0.95088,mine,0.95088,0.234%,7.37%,81.07%,11.56%
public:0.95081,public,0.95081,0.231%,7.39%,81.10%,11.51%
public:0.95095,public,0.95095,0.226%,7.38%,81.12%,11.50%
public:0.95113,public,0.95113,0.201%,7.38%,81.07%,11.55%
public:0.95112,public,0.95112,0.201%,7.38%,81.07%,11.55%


### 4. Visual analysis

Charts use inline SVG, so they render consistently without plotting-library setup. Darker heatmap cells mean higher pairwise agreement.

In [5]:
class BlendVisualizer:
    """Renders compact styled SVG diagnostics with no plotting dependency."""

    def __init__(self, width: int = 1080):
        self.width = width
        self.ink = "#0f172a"
        self.muted = "#64748b"
        self.border = "#dbe3ef"
        self.blue = "#2563eb"
        self.green = "#10b981"
        self.orange = "#f59e0b"

    def _wrap(self, title: str, subtitle: str, body: str) -> HTML:
        """Wraps a chart in a consistent report card."""

        markup = (
            f'<div style="font-family:-apple-system,BlinkMacSystemFont,Segoe UI,sans-serif;border:1px solid {self.border};'
            f'border-radius:12px;padding:18px;margin:14px 0;background:#fff">'
            f'<div style="font-size:19px;font-weight:750;color:{self.ink}">{html.escape(title)}</div>'
            f'<div style="font-size:13px;color:{self.muted};margin:5px 0 14px">{html.escape(subtitle)}</div>{body}</div>'
        )
        return HTML(markup)

    def bars(self, values: pd.Series, title: str, subtitle: str, percent: bool = False) -> HTML:
        """Draws ranked horizontal bars."""

        values = values.astype(float)
        maximum = max(float(values.max()), 1e-12)
        left, row_height, chart_width = 235, 34, self.width - 360
        parts = []
        for index, (label, value) in enumerate(values.items()):
            y = 16 + index * row_height
            width = chart_width * value / maximum
            shown = f"{value:.3%}" if percent else f"{value:,.0f}"
            parts.extend(
                [
                    f'<text x="0" y="{y+20}" font-size="11" fill="{self.ink}">{html.escape(str(label))}</text>',
                    f'<rect x="{left}" y="{y+6}" width="{width:.1f}" height="18" rx="5" fill="{self.blue}" opacity=".82"></rect>',
                    f'<text x="{left+width+8:.1f}" y="{y+20}" font-size="11" fill="{self.ink}">{shown}</text>',
                ]
            )
        height = 32 + len(values) * row_height
        return self._wrap(title, subtitle, f'<svg viewBox="0 0 {self.width} {height}" width="100%">{"".join(parts)}</svg>')

    def heatmap(self, matrix: pd.DataFrame, title: str, subtitle: str) -> HTML:
        """Draws an annotated agreement matrix."""

        left, top, cell = 205, 180, 72
        height = top + cell * len(matrix) + 25
        width = left + cell * len(matrix) + 20
        parts = []
        for column, label in enumerate(matrix.columns):
            x = left + column * cell + cell / 2
            parts.append(f'<text transform="translate({x:.1f},{top-10}) rotate(-55)" text-anchor="end" font-size="10" fill="{self.ink}">{html.escape(str(label))}</text>')
        for row, (label, values) in enumerate(matrix.iterrows()):
            y = top + row * cell
            parts.append(f'<text x="0" y="{y+42}" font-size="10" fill="{self.ink}">{html.escape(str(label))}</text>')
            for column, value in enumerate(values):
                alpha = 0.12 + 0.80 * max(0.0, min(1.0, (float(value) - 0.99) / 0.01))
                x = left + column * cell
                parts.extend(
                    [
                        f'<rect x="{x}" y="{y}" width="{cell-3}" height="{cell-3}" rx="6" fill="rgba(37,99,235,{alpha:.2f})"></rect>',
                        f'<text x="{x+cell/2}" y="{y+40}" text-anchor="middle" font-size="9" fill="{self.ink}">{float(value):.3%}</text>',
                    ]
                )
        return self._wrap(title, subtitle, f'<svg viewBox="0 0 {width} {height}" width="100%">{"".join(parts)}</svg>')

    def score_diversity(self, frame: pd.DataFrame) -> HTML:
        """Plots public score against diversity and labels every candidate."""

        plot_width, plot_height, left, top = 690, 330, 95, 25
        scores = frame["public_score"].astype(float)
        diversity = frame["diversity"].astype(float)
        score_min, score_max = float(scores.min()), float(scores.max())
        div_min, div_max = float(diversity.min()), float(diversity.max())
        score_span = max(score_max - score_min, 1e-9)
        div_span = max(div_max - div_min, 1e-9)
        parts = [
            f'<line x1="{left}" y1="{top+plot_height}" x2="{left+plot_width}" y2="{top+plot_height}" stroke="#94a3b8"></line>',
            f'<line x1="{left}" y1="{top}" x2="{left}" y2="{top+plot_height}" stroke="#94a3b8"></line>',
            f'<text x="{left+plot_width/2}" y="{top+plot_height+45}" text-anchor="middle" font-size="12" fill="{self.muted}">Public score</text>',
            f'<text transform="translate(22,{top+plot_height/2}) rotate(-90)" text-anchor="middle" font-size="12" fill="{self.muted}">Diversity</text>',
        ]
        for name, row in frame.iterrows():
            x = left + plot_width * (float(row["public_score"]) - score_min) / score_span
            y = top + plot_height * (1 - (float(row["diversity"]) - div_min) / div_span)
            color = self.green if row["source"] == "mine" else self.blue
            parts.extend(
                [
                    f'<circle cx="{x:.1f}" cy="{y:.1f}" r="7" fill="{color}" opacity=".85"></circle>',
                    f'<text x="{x+10:.1f}" y="{y-7:.1f}" font-size="9" fill="{self.ink}">{html.escape(str(name))}</text>',
                ]
            )
        return self._wrap("Score versus diversity", "Upper-right candidates combine leaderboard strength with a less duplicated signal.", f'<svg viewBox="0 0 {self.width} 420" width="100%">{"".join(parts)}</svg>')




class PremiumBlendVisualizer(BlendVisualizer):
    """Adds higher-contrast diagnostics while preserving the dependency-free SVG output."""

    def _wrap(self, title: str, subtitle: str, body: str) -> HTML:
        """Wraps charts in a restrained blue report card."""

        markup = (
            f'<div style="font-family:Inter,-apple-system,BlinkMacSystemFont,Segoe UI,sans-serif;border:1px solid #dbeafe;'
            f'border-radius:16px;padding:20px;margin:16px 0;background:linear-gradient(145deg,#ffffff 0%,#f8fbff 100%);'
            f'box-shadow:0 10px 28px rgba(15,23,42,.07)">'
            f'<div style="font-size:20px;font-weight:760;color:{self.ink};letter-spacing:-.2px">{html.escape(title)}</div>'
            f'<div style="font-size:13px;color:{self.muted};margin:6px 0 16px">{html.escape(subtitle)}</div>{body}</div>'
        )
        return HTML(markup)

    def disagreement_heatmap(self, agreement: pd.DataFrame) -> HTML:
        """Shows pairwise disagreement in basis points for useful contrast near 100 percent agreement."""

        matrix = (1 - agreement) * 10_000
        left, top, cell = 205, 180, 72
        height = top + cell * len(matrix) + 25
        width = left + cell * len(matrix) + 20
        maximum = max(float(matrix.to_numpy().max()), 1.0)
        parts = []
        for column, label in enumerate(matrix.columns):
            x = left + column * cell + cell / 2
            parts.append(f'<text transform="translate({x:.1f},{top-10}) rotate(-55)" text-anchor="end" font-size="10" fill="{self.ink}">{html.escape(str(label))}</text>')
        for row, (label, values) in enumerate(matrix.iterrows()):
            y = top + row * cell
            parts.append(f'<text x="0" y="{y+42}" font-size="10" fill="{self.ink}">{html.escape(str(label))}</text>')
            for column, value in enumerate(values):
                intensity = float(value) / maximum
                x = left + column * cell
                text_color = "#ffffff" if intensity > 0.58 else self.ink
                parts.extend(
                    [
                        f'<rect x="{x}" y="{y}" width="{cell-3}" height="{cell-3}" rx="8" fill="rgba(37,99,235,{0.06+0.88*intensity:.2f})"></rect>',
                        f'<text x="{x+cell/2}" y="{y+40}" text-anchor="middle" font-size="10" font-weight="650" fill="{text_color}">{float(value):.1f}</text>',
                    ]
                )
        return self._wrap("Pairwise disagreement", "Basis points of rows with different labels. Dark cells add more independent signal.", f'<svg viewBox="0 0 {width} {height}" width="100%">{"".join(parts)}</svg>')

    def class_composition(self, frame: pd.DataFrame) -> HTML:
        """Draws compact one-hundred-percent stacked bars for class balance."""

        colors = {"fit": "#10b981", "at-risk": "#3b82f6", "unhealthy": "#f97316"}
        left, top, row_height, chart_width = 205, 36, 34, self.width - 265
        parts = []
        for index, (name, row) in enumerate(frame.iterrows()):
            y, cursor = top + index * row_height, left
            parts.append(f'<text x="0" y="{y+17}" font-size="10" fill="{self.ink}">{html.escape(str(name))}</text>')
            for label in ("fit", "at-risk", "unhealthy"):
                value = float(row[f"share_{label}"])
                width = chart_width * value
                parts.append(f'<rect x="{cursor:.1f}" y="{y+3}" width="{width:.1f}" height="18" rx="4" fill="{colors[label]}"></rect>')
                cursor += width
        legend = "".join(f'<circle cx="{left+i*145}" cy="17" r="5" fill="{colors[label]}"></circle><text x="{left+10+i*145}" y="21" font-size="11" fill="{self.muted}">{label}</text>' for i, label in enumerate(colors))
        height = top + len(frame) * row_height + 12
        return self._wrap("Class composition", "Stable class shares indicate that no candidate shifts the dominant class globally.", f'<svg viewBox="0 0 {self.width} {height}" width="100%">{legend}{"".join(parts)}</svg>')

    def weight_heatmap(self, weights: pd.DataFrame) -> HTML:
        """Shows normalized candidate influence across base blend strategies."""

        matrix = weights.T
        left, top, cell_width, cell_height = 180, 180, 76, 42
        width = left + cell_width * len(matrix.columns) + 20
        height = top + cell_height * len(matrix) + 20
        maximum = max(float(matrix.to_numpy().max()), 1e-9)
        parts = []
        for column, label in enumerate(matrix.columns):
            x = left + column * cell_width + cell_width / 2
            parts.append(f'<text transform="translate({x:.1f},{top-10}) rotate(-55)" text-anchor="end" font-size="10" fill="{self.ink}">{html.escape(str(label))}</text>')
        for row, (label, values) in enumerate(matrix.iterrows()):
            y = top + row * cell_height
            parts.append(f'<text x="0" y="{y+26}" font-size="11" fill="{self.ink}">{html.escape(str(label))}</text>')
            for column, value in enumerate(values):
                intensity = float(value) / maximum
                x = left + column * cell_width
                parts.extend(
                    [
                        f'<rect x="{x}" y="{y}" width="{cell_width-3}" height="{cell_height-3}" rx="7" fill="rgba(16,185,129,{0.08+0.82*intensity:.2f})"></rect>',
                        f'<text x="{x+cell_width/2}" y="{y+25}" text-anchor="middle" font-size="10" fill="{self.ink}">{float(value):.3f}</text>',
                    ]
                )
        return self._wrap("Normalized strategy weights", "Each column candidate receives a different influence under each strategy.", f'<svg viewBox="0 0 {width} {height}" width="100%">{"".join(parts)}</svg>')

    def candidate_impact(self, report: pd.DataFrame) -> HTML:
        """Ranks unique output candidates by changed rows versus the strongest anchor."""

        exported = report.loc[report["exported"]].set_index("blend_name")["changed_rows_vs_best"].sort_values(ascending=False)
        return self.bars(exported, "Candidate impact", "Rows changed relative to the strongest public-score anchor. More changes mean more risk and diversity.")


visualizer = PremiumBlendVisualizer(config.svg_width)
display(visualizer.disagreement_heatmap(agreement))
display(visualizer.class_composition(summary))
display(visualizer.score_diversity(summary))
display(visualizer.bars(vote_strength, "Vote coalition size", "Largest same-label coalition across all candidates. Low values identify uncertain rows."))

### 5. Build transparent blends

Base strategies cover equal, score, diversity, hybrid, and cluster-balanced voting. Three additional anchor strategies change the strongest public submission only when independent model families reach 75%, 65%, or 55% weighted consensus. Prediction-identical outputs are reported but not exported twice.

In [6]:
class SubmissionBlender:
    """Creates deterministic weighted votes from aligned hard labels."""

    def __init__(self, config: BlendConfig, analyzer: SubmissionAnalyzer, summary: pd.DataFrame, agreement: pd.DataFrame):
        self.config = config
        self.analyzer = analyzer
        self.summary = summary.reindex(analyzer.predictions.columns)
        self.agreement = agreement.reindex(index=analyzer.predictions.columns, columns=analyzer.predictions.columns)
        self.names = analyzer.predictions.columns.tolist()
        self.label_to_code = {label: index for index, label in enumerate(config.labels)}
        self.encoded = analyzer.predictions.apply(lambda column: column.map(self.label_to_code)).to_numpy(dtype=np.int8)

    def _score_weights(self) -> pd.Series:
        """Applies a bounded linear preference around the median public score."""

        scores = self.summary["public_score"].fillna(self.summary["public_score"].median())
        return (1 + (scores - scores.median()) * self.config.score_weight_scale).clip(0.5, 1.5)

    def _diversity_weights(self) -> pd.Series:
        """Rewards non-duplicate signal while keeping weights bounded."""

        diversity = self.summary["diversity"].clip(lower=1e-9)
        return (diversity / diversity.mean()).clip(0.5, 2.0)

    def _duplicate_groups(self) -> list[list[str]]:
        """Finds connected groups above the near-duplicate agreement threshold."""

        parent = list(range(len(self.names)))

        def find(index: int) -> int:
            while parent[index] != index:
                parent[index] = parent[parent[index]]
                index = parent[index]
            return index

        def union(left: int, right: int) -> None:
            root_left, root_right = find(left), find(right)
            if root_left != root_right:
                parent[root_right] = root_left

        for left in range(len(self.names)):
            for right in range(left + 1, len(self.names)):
                if self.agreement.iloc[left, right] >= self.config.near_duplicate_threshold:
                    union(left, right)
        groups = {}
        for index, name in enumerate(self.names):
            groups.setdefault(find(index), []).append(name)
        return list(groups.values())

    def weight_table(self) -> pd.DataFrame:
        """Returns auditable weights for every strategy and candidate."""

        score = self._score_weights()
        diversity = self._diversity_weights()
        cluster = pd.Series(1.0, index=self.names)
        for group in self._duplicate_groups():
            cluster.loc[group] = 1 / len(group)
        weights = pd.DataFrame(
            {
                "majority": 1.0,
                "score_weighted": score,
                "diversity_weighted": diversity,
                "hybrid": score * diversity,
                "cluster_balanced": cluster,
            },
            index=self.names,
        )
        return weights.apply(lambda column: column / column.sum(), axis=0)

    def blend(self, weights: pd.Series) -> tuple[pd.DataFrame, pd.Series]:
        """Produces labels and normalized winning margins with score-based tie breaking."""

        weight_values = weights.reindex(self.names).to_numpy(dtype=float)
        totals = np.column_stack(
            [(self.encoded == code).astype(float) @ weight_values for code in range(len(self.config.labels))]
        )
        winners = totals.argmax(axis=1)
        ties = (totals == totals.max(axis=1, keepdims=True)).sum(axis=1) > 1
        if ties.any():
            best_name = self.summary["public_score"].idxmax()
            best_codes = self.encoded[:, self.names.index(best_name)]
            allowed = totals[np.arange(len(totals)), best_codes] == totals.max(axis=1)
            winners[ties & allowed] = best_codes[ties & allowed]
        labels = np.array(self.config.labels, dtype=object)[winners]
        margin = pd.Series(totals.max(axis=1) / totals.sum(axis=1), name="winning_vote_share")
        result = pd.DataFrame({self.config.id_column: self.analyzer.ids, self.config.target: labels})
        return result, margin

    def export_all(self, output: Path) -> tuple[pd.DataFrame, dict[str, pd.DataFrame]]:
        """Exports one competition-ready CSV per strategy and a compact report."""

        output.mkdir(parents=True, exist_ok=True)
        weights = self.weight_table()
        blends = {}
        records = []
        reference_name = self.summary["public_score"].idxmax()
        reference = self.analyzer.predictions[reference_name].to_numpy()
        for strategy in weights.columns:
            blend, margin = self.blend(weights[strategy])
            blend.to_csv(output / f"blend_{strategy}.csv", index=False)
            blends[strategy] = blend
            records.append(
                {
                    "blend_name": strategy,
                    "changed_rows_vs_best": int(np.sum(blend[self.config.target].to_numpy() != reference)),
                    "agreement_with_best": float(np.mean(blend[self.config.target].to_numpy() == reference)),
                    "mean_winning_vote_share": float(margin.mean()),
                    "minimum_winning_vote_share": float(margin.min()),
                }
            )
        report = pd.DataFrame(records)
        report.to_csv(output / "blend_report.csv", index=False)
        weights.to_csv(output / "blend_weights.csv", index_label="submission")
        return report, blends




class FamilyAwareBlender(SubmissionBlender):
    """Adds anchor overrides and exports only prediction-distinct candidates."""

    def _family_predictions(self) -> tuple[np.ndarray, list[list[str]]]:
        """Collapses each near-duplicate family into one hard-label vote."""

        groups = self._duplicate_groups()
        family_columns = []
        for group in groups:
            indices = [self.names.index(name) for name in group]
            family_values = self.encoded[:, indices]
            totals = np.column_stack([(family_values == code).sum(axis=1) for code in range(len(self.config.labels))])
            winners = totals.argmax(axis=1)
            ties = (totals == totals.max(axis=1, keepdims=True)).sum(axis=1) > 1
            if ties.any():
                best_name = self.summary.loc[group, "public_score"].idxmax()
                winners[ties] = self.encoded[ties, self.names.index(best_name)]
            family_columns.append(winners)
        return np.column_stack(family_columns), groups

    def anchor_override(self, threshold: float) -> tuple[pd.DataFrame, pd.Series]:
        """Changes the strongest anchor only under broad independent-family agreement."""

        family_predictions, groups = self._family_predictions()
        family_scores = pd.Series([self.summary.loc[group, "public_score"].max() for group in groups])
        family_weights = (1 + (family_scores - family_scores.median()) * self.config.score_weight_scale).clip(0.5, 1.5).to_numpy()
        totals = np.column_stack([(family_predictions == code).astype(float) @ family_weights for code in range(len(self.config.labels))])
        challengers = totals.argmax(axis=1)
        confidence = totals.max(axis=1) / totals.sum(axis=1)
        support = np.column_stack([(family_predictions == code).sum(axis=1) for code in range(len(self.config.labels))])
        challenger_support = support[np.arange(len(support)), challengers]
        anchor_name = self.summary["public_score"].idxmax()
        anchor_codes = self.encoded[:, self.names.index(anchor_name)]
        change = (challengers != anchor_codes) & (confidence >= threshold) & (challenger_support >= 3)
        result_codes = anchor_codes.copy()
        result_codes[change] = challengers[change]
        labels = np.array(self.config.labels, dtype=object)[result_codes]
        result = pd.DataFrame({self.config.id_column: self.analyzer.ids, self.config.target: labels})
        return result, pd.Series(confidence, name="winning_vote_share")

    def export_all(self, output: Path) -> tuple[pd.DataFrame, dict[str, pd.DataFrame]]:
        """Exports unique base and anchor candidates with an auditable shortlist report."""

        output.mkdir(parents=True, exist_ok=True)
        for path in output.glob("blend_*.csv"):
            path.unlink()
        weights = self.weight_table()
        candidates = {strategy: self.blend(weights[strategy]) for strategy in weights.columns}
        candidates.update(
            {
                "anchor_conservative": self.anchor_override(0.75),
                "anchor_balanced": self.anchor_override(0.65),
                "anchor_aggressive": self.anchor_override(0.55),
            }
        )
        anchor_name = self.summary["public_score"].idxmax()
        anchor = self.analyzer.predictions[anchor_name].to_numpy()
        unique_predictions = {f"anchor:{anchor_name}": anchor.copy()}
        exported_blends = {}
        records = []
        for strategy, (blend, margin) in candidates.items():
            labels = blend[self.config.target].to_numpy()
            duplicate_of = next((name for name, values in unique_predictions.items() if np.array_equal(labels, values)), "")
            exported = duplicate_of == ""
            file_name = f"blend_{strategy}.csv" if exported else ""
            if exported:
                blend.to_csv(output / file_name, index=False)
                unique_predictions[strategy] = labels.copy()
                exported_blends[strategy] = blend
            changed = int(np.sum(labels != anchor))
            risk = "low" if changed <= 75 else "medium" if changed <= 200 else "high"
            records.append(
                {
                    "blend_name": strategy,
                    "exported": exported,
                    "duplicate_of": duplicate_of,
                    "risk": risk,
                    "changed_rows_vs_best": changed,
                    "agreement_with_best": float(np.mean(labels == anchor)),
                    "mean_winning_vote_share": float(margin.mean()),
                    "minimum_winning_vote_share": float(margin.min()),
                    "file_name": file_name,
                }
            )
        report = pd.DataFrame(records)
        report.to_csv(output / "blend_candidate_report.csv", index=False)
        weights.to_csv(output / "blend_weights.csv", index_label="submission")
        return report, exported_blends


blender = FamilyAwareBlender(config, analyzer, summary, agreement)
weight_table = blender.weight_table()
blend_report, blends = blender.export_all(repository.output)
report_view = blend_report[["blend_name", "exported", "duplicate_of", "risk", "changed_rows_vs_best", "agreement_with_best"]]
display(report_view.style.format({"agreement_with_best": "{:.3%}"}).hide(axis="index"))
display(visualizer.weight_heatmap(weight_table))
display(visualizer.candidate_impact(blend_report))
print(f"Saved unique blend artifacts to: {repository.output.resolve()}")

blend_name,exported,duplicate_of,risk,changed_rows_vs_best,agreement_with_best
majority,True,,low,43,99.985%
score_weighted,False,majority,low,43,99.985%
diversity_weighted,True,,high,292,99.901%
hybrid,True,,medium,103,99.965%
cluster_balanced,True,,high,228,99.923%
anchor_conservative,False,anchor:public:0.95114,low,0,100.000%
anchor_balanced,True,,low,37,99.987%
anchor_aggressive,True,,high,225,99.924%


Saved unique blend artifacts to: /Users/flexonafft/MLKaggleTasks/competitions/monthly-comp/predicting-student-health-risk/health-submiss-pool/blends


### 6. Verify generated submissions

Every exported blend passes the same schema and identifier checks as its inputs. This is the smallest runnable guard against malformed Kaggle files.

In [7]:
generated_quality = validator.validate({f"blend:{name}": frame for name, frame in blends.items()})
assert len(blends) >= 5
assert generated_quality["schema_valid"].all()
assert generated_quality["id_order_aligned"].all()
assert all((repository.output / f"blend_{name}.csv").exists() for name in blends)
generated_view = generated_quality[["name", "rows"]].rename(columns={"name": "candidate"})
generated_view["status"] = "ready"
display(generated_view.style.hide(axis="index"))

candidate,rows,status
blend:majority,295753,ready
blend:diversity_weighted,295753,ready
blend:hybrid,295753,ready
blend:cluster_balanced,295753,ready
blend:anchor_balanced,295753,ready
blend:anchor_aggressive,295753,ready


## Takeaways

- Prefer candidates that balance score and diversity, not score alone.
- Treat near-identical files as one prediction family.
- Submit a small set of meaningfully different blends rather than many tiny weight variants.
- Upgrade to probability blending only when raw class probabilities or OOF predictions become available.

In [8]:
class NotebookSummary:
    """Prints final observed facts after every calculation succeeds."""

    @staticmethod
    def show(catalog: pd.DataFrame, agreement: pd.DataFrame, vote_strength: pd.Series, report: pd.DataFrame) -> None:
        """Reports pool size, disagreement, duplicate intensity, and generated artifacts."""

        off_diagonal = agreement.to_numpy()[~np.eye(len(agreement), dtype=bool)]
        total_rows = len(analyzer.predictions)
        unanimous_rows = int(vote_strength.get(len(catalog), 0))
        print(f"Candidates: {len(catalog)}")
        print(f"Rows: {total_rows:,}")
        print(f"Unanimous rows: {unanimous_rows:,} ({unanimous_rows / total_rows:.3%})")
        print(f"Pairwise agreement range: {off_diagonal.min():.3%} to {off_diagonal.max():.3%}")
        print(f"Generated unique blends: {', '.join(report.loc[report['exported'], 'blend_name'])}")


NotebookSummary.show(catalog, agreement, vote_strength, blend_report)

Candidates: 11
Rows: 295,753
Unanimous rows: 293,183 (99.131%)
Pairwise agreement range: 99.319% to 100.000%
Generated unique blends: majority, diversity_weighted, hybrid, cluster_balanced, anchor_balanced, anchor_aggressive
